# Unit 14 / Chapter 14: Quantum Federated and Privacy-Preserving Learning

> **Main Learning Objective:** Understand classical federated learning, see where privacy breaks, and learn how quantum primitives (blind quantum computing, secure aggregation, quantum differential privacy) offer stronger guarantees. Simulate a small quantum federated training round.

| Section | Topic |
|---|---|
| 14.1 | Federated learning basics and their privacy holes |
| 14.2 | Blind quantum computing intuition |
| 14.3 | Quantum secure aggregation |
| 14.4 | Simulating a federated quantum training round |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 14**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 14
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 14.1: Federated Learning Basics and Their Privacy Holes

**Federated learning** (FL) trains a shared model across many clients (phones, hospitals, banks) without ever sending the raw data to a central server. Each client trains locally, then sends only its model updates (gradients or weights) to the server, which averages them.

Google's Gboard is the classic example: each phone trains a next-word model locally, and only the weight updates fly to the server.

The problem: **updates leak data**. Multiple papers have shown that seeing a client's gradient can reveal their training images or text with high fidelity (Zhu et al. 2019, "Deep Leakage from Gradients"). So "we never send raw data" is not the same as "the data is private."

Standard classical fixes:

* **Secure aggregation**: cryptographic protocols so the server only sees the *sum* of updates, not any individual one.
* **Differential privacy**: adding calibrated noise to each update.

Both work, but they have overhead and require careful key management. Enter quantum.

---
# Section 14.2: Blind Quantum Computing Intuition

**Blind quantum computing (BQC)** lets a client with almost no quantum hardware delegate a quantum computation to a powerful server, in a way that the server learns *nothing* about the input, the computation, or the output.

The core trick, from the Universal Blind Quantum Computation protocol by Broadbent, Fitzsimons, and Kashefi (2009):

1. The client sends the server random single-qubit states.
2. The client tells the server which measurements to make, in an order that only makes sense given the (secret) computation the client is running.
3. The server does the measurements and reports outcomes back.
4. The client interprets the outcomes.

At every step, the server sees only random-looking states and instructions. It cannot infer the actual computation.

In FL terms: each client can compute a gradient on a quantum server without revealing their data. That closes the "gradient leaks data" hole at its root.

---
# Section 14.3: Quantum Secure Aggregation

Even without full BQC, a lighter-weight quantum primitive helps: **quantum secure aggregation**.

Idea in one line: encode each client's update as a rotation on a shared entangled state. Only the *sum* of rotations produces a coherent measurement outcome. Any individual client's rotation looks random on its own.

Below we simulate this: each client has a scalar update (say a single gradient value). We encode each into a rotation, apply them all on the same qubit, and show that only the sum matters when we measure.

In [ ]:
def secure_aggregate(updates):
    n = 1
    s = ket0(n)
    # Apply each client's Ry rotation on the same qubit
    for u in updates:
        s = apply_1q(Ry(u), 0, n) @ s
    # Measure <Z>. cos(sum of thetas) if all Ry composed
    return expZ(s, 0, n)

updates_A = [0.3, 0.4, 0.5]
updates_B = [0.5, 0.3, 0.4]  # same total, different individuals
z_A = secure_aggregate(updates_A)
z_B = secure_aggregate(updates_B)
print("aggregate for group A:", round(z_A, 4), "  sum:", round(sum(updates_A), 4))
print("aggregate for group B:", round(z_B, 4), "  sum:", round(sum(updates_B), 4))
print("same aggregate for same sum:", np.isclose(z_A, z_B))

The measurement output is the same for both groups because it only depends on the sum of the individual updates. A server that sees only <Z> cannot tell what any single client contributed.

### Activity 14.1

Add a fourth client with update 0.2 to group A. What do you expect the aggregate to be, and does the code confirm it?

In [ ]:
# TODO: extend group A with one more client and re-aggregate.
updates_A_new = None       # <-- list of 4 numbers

if updates_A_new is not None:
    print("new aggregate:", round(secure_aggregate(updates_A_new), 4))
    print("expected cos(sum/2 * 2) via <Z>: cos(sum(updates)):",
          round(np.cos(sum(updates_A_new)), 4))
else:
    print("Fill in updates_A_new, then re-run.")

<details><summary>Solution</summary>

```python
updates_A_new = [0.3, 0.4, 0.5, 0.2]
```
The aggregate is cos(1.4) ~ 0.170. Adding a client shifts the aggregate but reveals only the sum, not individual values.
</details>

---
# Section 14.4: A Simulated Federated Quantum Training Round

Below we simulate one round of quantum federated learning:

1. Three clients each hold two labeled samples.
2. Each client runs a local variational quantum classifier and computes its local gradient.
3. The server aggregates the gradients via quantum secure aggregation.
4. The global model updates.

We only run one round for clarity.

In [ ]:
client_data = [
    [((0.1, 0.2), +1), ((0.15, 0.1), +1)],
    [((0.9, 1.0), -1), ((1.0, 0.9), -1)],
    [((0.5, 0.5),  0), ((0.55, 0.45), 0)],   # neutral center
]

def local_model(theta, x):
    s = ket0(1)
    s = Ry(x[0] + theta) @ s
    return expZ(s.reshape(-1), 0, 1)  # 1-qubit expZ

def local_gradient(theta, samples, eps=0.1):
    def L(t):
        return sum((local_model(t, x) - y)**2 for x, y in samples) / len(samples)
    return (L(theta+eps) - L(theta-eps)) / (2*eps)

theta = 0.4
grads = [local_gradient(theta, cd) for cd in client_data]
print("local gradients:", np.round(grads, 3))

agg_grad_signal = secure_aggregate([g/10 for g in grads])   # server sees only agg
print("aggregated signal (server sees only this):", round(agg_grad_signal, 3))

### Activity 14.2

Suppose one client (say client 0) drops out. Re-aggregate using only clients 1 and 2. Does the aggregated signal change? Why is that a problem for FL and how do you fix it?

In [ ]:
# TODO: aggregate only clients 1 and 2.
grads_partial = None       # <-- [grads[1], grads[2]]

if grads_partial is not None:
    partial_agg = secure_aggregate([g/10 for g in grads_partial])
    print("partial aggregate (server sees):", round(partial_agg, 3))
    print("This differs from the full aggregate, which is fine as long as the")
    print("server knows the number of participating clients. If the count is")
    print("hidden too, the mean estimate is biased. Practical FL protocols")
    print("solve this with a public participation count.")
else:
    print("Fill in grads_partial, then re-run.")

<details><summary>Solution</summary>

```python
grads_partial = [grads[1], grads[2]]
```
The aggregate changes because the sum changes. In practice, servers know how many clients contributed (that count is public), so they can rescale the aggregate. The privacy guarantee is about the *values*, not the *count*.
</details>

---
## Section summary

* Federated learning trains without moving raw data, but gradient updates leak information.
* Blind quantum computing lets clients delegate quantum work while hiding the data, the computation, and the output.
* Quantum secure aggregation reveals only sums of client updates, protecting individuals.
* One simulated federated quantum round chains local gradients through secure aggregation to update a global model.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** Federated learning keeps data on-device but shares:

A. Raw images
B. Model updates like gradients or weights
C. Nothing
D. Only user IDs

**Q2.** The 2019 "Deep Leakage from Gradients" paper showed that:

A. Gradients cannot be inverted
B. Gradients can leak training data with high fidelity
C. Federated learning is impossible
D. Only vision models leak

**Q3.** Secure aggregation ensures the server only sees:

A. The largest update
B. The sum of updates, not individuals
C. A random subset
D. The client IDs

**Q4.** Blind quantum computing hides which of the following from the server?

A. The number of qubits
B. The input, the computation, and the output
C. Only the input
D. Only the output

**Q5.** In our quantum secure aggregation demo, the measurement outcome depended on:

A. The maximum client update
B. The sum of the individual Ry angles
C. The variance of the updates
D. The number of qubits

**Q6.** What breaks if a client drops out mid-round of secure aggregation?

A. Nothing, aggregation is invariant
B. The sum shifts, so the server needs a public count to unbias
C. The whole protocol crashes
D. All privacy is lost

**Q7.** Differential privacy adds:

A. Encryption keys
B. Calibrated noise to each update
C. Extra qubits
D. Nothing, it is a legal concept

**Q8.** Compared to classical secure aggregation, quantum secure aggregation:

A. Removes the need for cryptographic key management for certain aggregate primitives
B. Is always slower
C. Requires no server at all
D. Works only on integer data

**Q9.** The BQC protocol was proposed by:

A. Shor, Grover, and Feynman
B. Broadbent, Fitzsimons, and Kashefi
C. Rivest, Shamir, and Adleman
D. Deutsch and Jozsa

**Q10.** The practical draw of quantum federated learning is:

A. It replaces neural networks
B. It provides privacy guarantees rooted in physics rather than cryptographic assumptions
C. It requires no math
D. It runs faster on today's laptops

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 14!")